In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Load data
df = pd.read_excel("GuttmacherInstituteAbortionDataByState.xlsx", sheet_name="Guttmacher")

# -----------------------------
# Political lean mapping
# -----------------------------
democrat_states = [
    "California", "Colorado", "Connecticut", "Delaware", "Hawaii",
    "Illinois", "Maine", "Maryland", "Massachusetts", "Minnesota",
    "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York",
    "Oregon", "Rhode Island", "Vermont", "Virginia", "Washington",
    "District of Columbia"
]

republican_states = [
    "Alabama", "Alaska", "Arkansas", "Florida", "Idaho", "Indiana",
    "Iowa", "Kansas", "Kentucky", "Louisiana", "Mississippi", "Missouri",
    "Montana", "Nebraska", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "South Carolina", "South Dakota", "Tennessee", "Texas",
    "Utah", "West Virginia", "Wyoming"
]

swing_states = [
    "Arizona", "Georgia", "Michigan", "Pennsylvania", "Wisconsin"
]

party_map = (
    {s: "Democrat" for s in democrat_states} |
    {s: "Republican" for s in republican_states} |
    {s: "Swing" for s in swing_states}
)

df["political_lean"] = df["U.S. State"].map(party_map)

color_map = {
    "Democrat": "#1f77b4",
    "Republican": "#d62728",
    "Swing": "#9467bd"
}

# -----------------------------
# Numeric cleanup
# -----------------------------
numeric_cols = [
    "% of women aged 15-44 living in a county without a clinic, 2020",
    "% of residents obtaining abortions who traveled out of state for care, 2020",
    "No. of abortions, by state of occurrence, 2020",
    "No. of abortions, by state of residence, 2020",
    "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Net flow: positive = importer, negative = exporter
df["net_flow"] = (
    df["No. of abortions, by state of occurrence, 2020"]
    - df["No. of abortions, by state of residence, 2020"]
)

In [2]:
def apply_clean_layout(fig, width=980, height=620, show_legend=True, bottom_margin=90):
    fig.update_layout(
        width=width,
        height=height,
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(size=13),
        title=dict(
            x=0.5,
            xanchor="center",
            font=dict(size=20)
        ),
        margin=dict(t=90, l=80, r=40, b=bottom_margin),
        showlegend=show_legend,
        legend=dict(
            x=0.98,
            y=0.98,
            xanchor="right",
            yanchor="top",
            orientation="v",
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#d9d9d9",
            borderwidth=1,
            font=dict(size=11),
            title_font=dict(size=12)
        )
    )
    return fig

In [3]:
col_x1 = "% of women aged 15-44 living in a county without a clinic, 2020"
col_y1 = "% of residents obtaining abortions who traveled out of state for care, 2020"

fig1 = px.scatter(
    df,
    x=col_x1,
    y=col_y1,
    color="political_lean",
    color_discrete_map=color_map,
    trendline="ols",
    hover_name="U.S. State",
    title="Clinic Deserts Force Out-of-State Travel",
    labels={
        col_x1: "% Women in Counties Without a Clinic",
        col_y1: "% Residents Traveling Out of State",
        "political_lean": "Political Lean"
    }
)

fig1.update_traces(marker=dict(size=7, opacity=0.8))

fig1.update_xaxes(
    range=[0, 100],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False,
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

fig1.update_yaxes(
    range=[0, 100],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False,
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

# apply base layout first
apply_clean_layout(fig1, width=980, height=620, show_legend=True, bottom_margin=80)

# override legend JUST for this plot
fig1.update_layout(
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="#d9d9d9",
        borderwidth=1,
        font=dict(size=11),
        title_font=dict(size=12)
    )
)

fig1.show()

In [4]:
df_flow = df.sort_values("net_flow", ascending=True).copy()

fig2 = px.bar(
    df_flow,
    x="U.S. State",
    y="net_flow",
    color="political_lean",
    color_discrete_map=color_map,
    hover_name="U.S. State",
    title="Net Abortion Flow by State",
    labels={
        "U.S. State": "State",
        "net_flow": "Net Flow (Occurrence - Residence)",
        "political_lean": "Political Lean"
    }
)

fig2.update_xaxes(
    categoryorder="array",
    categoryarray=df_flow["U.S. State"],
    tickangle=60,
    showgrid=False,
    title_font=dict(size=15),
    tickfont=dict(size=9)
)

fig2.update_yaxes(
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=True,
    zerolinecolor="black",
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

apply_clean_layout(fig2, width=1180, height=680, show_legend=True, bottom_margin=150)
fig2.show()

In [5]:
# Add subtitle via layout annotation (Plotly doesn't have native subtitles)
fig2.update_layout(
    title=dict(
        text="Net Abortion Flow by State<br><sup style='font-size:13px; color:gray'>Positive values indicate more abortions occurred in-state than residents sought; negative values show net outflow to other states (2020)</sup>",
        font=dict(size=20),
        x=0.5,
        xanchor="center"
    )
)

# Annotation 1: Highlight Missouri (largest negative bar — most residents leaving)
fig2.add_annotation(
    x="Missouri",
    y=-11500,
    text="Missouri: largest net outflow<br>after near-total ban",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#333",
    ax=60,
    ay=-40,
    font=dict(size=11, color="#333"),
    bgcolor="white",
    bordercolor="#999",
    borderwidth=1
)

# Annotation 2: Highlight Illinois (largest positive bar — destination state)
fig2.add_annotation(
    x="Illinois",
    y=10800,
    text="Illinois: top destination<br>for out-of-state patients",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#333",
    ax=-60,
    ay=-40,
    font=dict(size=11, color="#333"),
    bgcolor="white",
    bordercolor="#999",
    borderwidth=1
)

# High-res export (scale=3 gives ~3x pixel density, good for embedding)
fig2.write_image("net_abortion_flow_hires.png", scale=3, width=1400, height=700)

fig2.show()

In [6]:
# Remove old annotations first (if re-running)
fig2.layout.annotations = []

# Annotation 1: Importer/Exporter dividing line
# Find the approximate x-index where bars cross zero — around "Louisiana" or nearby
fig2.add_vline(
    x="Iowa",  # adjust to whichever state is the last negative bar
    line_dash="dash",
    line_color="#555",
    line_width=1.5,
    opacity=0.6
)

fig2.add_annotation(
    x="Iowa",
    y=9500,
    text="← Net Exporters (outflow) | Net Importers (inflow) →",
    showarrow=False,
    font=dict(size=11, color="#444"),
    bgcolor="white",
    bordercolor="#aaa",
    borderwidth=1,
    xanchor="center"
)

# Annotation 2: Red states = exporters
fig2.add_annotation(
    x="Texas",
    y=0,
    text="Republican-led states drive<br>most of the outflow",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#d62728",
    ax=0,
    ay=-50,
    font=dict(size=11, color="#d62728"),
    bgcolor="white",
    bordercolor="#d62728",
    borderwidth=1
)

# Annotation 3: Blue states = importers
fig2.add_annotation(
    x="New York",
    y=0,
    text="Democrat-led states absorb<br>the majority of inflow",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#1f77b4",
    ax=0,
    ay=50,
    font=dict(size=11, color="#1f77b4"),
    bgcolor="white",
    bordercolor="#1f77b4",
    borderwidth=1
)

In [7]:
df_flow = df.sort_values("net_flow", ascending=True).copy()

fig2 = px.bar(
    df_flow,
    x="U.S. State",
    y="net_flow",
    color="political_lean",
    color_discrete_map=color_map,
    hover_name="U.S. State",
    title="Net Abortion Flow by State",
    labels={
        "U.S. State": "State",
        "net_flow": "Net Flow (Occurrence - Residence)",
        "political_lean": "Political Lean"
    }
)

fig2.update_xaxes(
    categoryorder="array",
    categoryarray=df_flow["U.S. State"],
    tickangle=60,
    showgrid=False,
    title_font=dict(size=15),
    tickfont=dict(size=9)
)

fig2.update_yaxes(
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=True,
    zerolinecolor="black",
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

# Add subtitle via layout annotation (Plotly doesn't have native subtitles)
fig2.update_layout(
    title=dict(
        text="Net Abortion Flow by State<br><sup style='font-size:13px; color:gray'>Positive values indicate more abortions occurred in-state than residents sought; negative values show net outflow to other states (2020)</sup>",
        font=dict(size=20),
        x=0.5,
        xanchor="center"
    )
)

# Annotation 1: Importer/Exporter dividing line
fig2.add_vline(
    x="Iowa", 
    line_dash="dash",
    line_color="#555",
    line_width=1.5,
    opacity=0.6
)

fig2.add_annotation(
    x="Iowa",
    y=9500,
    text="← Net Exporters (outflow) | Net Importers (inflow) →",
    showarrow=False,
    font=dict(size=11, color="#444"),
    bgcolor="white",
    bordercolor="#aaa",
    borderwidth=1,
    xanchor="center"
)

# Annotation 2: Red states = exporters
fig2.add_annotation(
    x="Texas",
    y=0,
    text="Republican-led states drive<br>most of the outflow",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#d62728",
    ax=0,
    ay=-50,
    font=dict(size=11, color="#d62728"),
    bgcolor="white",
    bordercolor="#d62728",
    borderwidth=1
)

# Annotation 3: Blue states = importers
fig2.add_annotation(
    x="New York",
    y=0,
    text="Democrat-led states absorb<br>the majority of inflow",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#1f77b4",
    ax=0,
    ay=50,
    font=dict(size=11, color="#1f77b4"),
    bgcolor="white",
    bordercolor="#1f77b4",
    borderwidth=1
)

apply_clean_layout(fig2, width=1180, height=680, show_legend=True, bottom_margin=150)
fig2.write_image("net_abortion_flow_hires.png", scale=3, width=1400, height=700)
fig2.write_html("../net_abortion_flow.html", include_plotlyjs="cdn")

fig2.show()

In [8]:
col_y3 = "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"

df_box = df[df["political_lean"].isin(["Democrat", "Republican"])].copy()

fig3 = px.box(
    df_box,
    x="political_lean",
    y=col_y3,
    color="political_lean",
    color_discrete_map=color_map,
    points="all",
    category_orders={"political_lean": ["Democrat", "Republican"]},
    hover_name="U.S. State",
    title="Abortion Rate by Political Lean",
    labels={
        "political_lean": "Political Lean",
        col_y3: "Abortions per 1,000 Women Aged 15–44"
    }
)

fig3.update_traces(marker=dict(size=6, opacity=0.7))

fig3.update_xaxes(
    showgrid=False,
    title_font=dict(size=15),
    tickfont=dict(size=12)
)

fig3.update_yaxes(
    range=[0, 30],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False,
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

apply_clean_layout(fig3, width=1100, height=620, show_legend=False, bottom_margin=80)
fig3.update_layout(autosize=True)
fig3.show()


In [9]:

# Compute medians dynamically
dem_median = df_box[df_box["political_lean"] == "Democrat"][col_y3].median()
rep_median = df_box[df_box["political_lean"] == "Republican"][col_y3].median()
gap = dem_median - rep_median

# Updated title with subtitle clarifying "by residence"
fig3.update_layout(
    title=dict(
        text="Restrictive Policies Correlate with Lower Abortion Rates<br><sup style='font-size:12px; color:gray'>Abortions per 1,000 women aged 15–44, by state of residence — already accounts for out-of-state travel (2020)</sup>",
        font=dict(size=20),
        x=0.5,
        xanchor="center"
    )
)

# Dashed reference lines
fig3.add_hline(
    y=dem_median,
    line_dash="dash",
    line_color="#1f77b4",
    opacity=0.5,
    line_width=1.5,
    annotation_text=f"Democrat median: {dem_median:.1f}",
    annotation_position="top right",
    annotation_font=dict(size=10, color="#1f77b4")
)

fig3.add_hline(
    y=rep_median,
    line_dash="dash",
    line_color="#d62728",
    opacity=0.5,
    line_width=1.5
)

# Republican median label — matched to same right-edge position as Democrat label
fig3.add_annotation(
    x=1.0,
    xref="paper",
    y=rep_median + (gap * 0.25),
    yref="y",
    text=f"Republican median: {rep_median:.1f}",
    showarrow=False,
    font=dict(size=10, color="#d62728"),
    align="right",
    xanchor="right"
)

out_path = "../plot1against.html"
fig3.write_html(out_path, include_plotlyjs="cdn", config={"responsive": True})

# Strip browser default body margin so chart fills the iframe edge-to-edge
with open(out_path, "r", encoding="utf-8") as f:
    html = f.read()
html = html.replace("</head>", "<style>body{margin:0;padding:0;}</style></head>", 1)
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html)

fig3.show()


In [10]:
col_x4 = "% of residents obtaining abortions who traveled out of state for care, 2020"
col_y4 = "No. of abortions per 1,000 women aged 15–44, by state of residence, 2020"

df_anti = df[df["political_lean"].isin(["Democrat", "Republican"])].copy()
df_anti = df_anti[df_anti[col_y4] <= 20].copy()

fig4 = px.scatter(
    df_anti,
    x=col_x4,
    y=col_y4,
    color="political_lean",
    color_discrete_map={
        "Republican": "#d62728",
        "Democrat": "#1f77b4"
    },
    trendline="ols",
    hover_name="U.S. State",
    title="Out-of-State Travel vs Abortion Rate",
    labels={
        col_x4: "% Residents Traveling Out of State",
        col_y4: "Abortions per 1,000 Women Aged 15–44",
        "political_lean": "Political Lean"
    }
)

fig4.update_traces(marker=dict(size=7, opacity=0.82))

fig4.update_xaxes(
    range=[0, 100],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False,
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

fig4.update_yaxes(
    range=[0, 20],
    showgrid=True,
    gridcolor="#e5e5e5",
    zeroline=False,
    title_font=dict(size=15),
    tickfont=dict(size=11)
)

apply_clean_layout(fig4, width=980, height=620, show_legend=True, bottom_margin=80)
fig4.show()